<a href="https://colab.research.google.com/github/asdp132A3a/alt-tab-macos/blob/master/fsrs4anki_optimizer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# FSRS4Anki v6.1.3 Optimizer

[![open in colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/open-spaced-repetition/fsrs4anki/blob/v6.1.3/fsrs4anki_optimizer.ipynb)

↑ Click the above button to open the optimizer on Google Colab.

> If you can't see the button and are located in the Chinese Mainland, please use a proxy or VPN.

Upload your **Anki Deck Package (.apkg)** file or **Anki Collection Package (.colpkg)** file on the `Left sidebar -> Files`, drag and drop your file in the current directory (not the `sample_data` directory).

No need to include media. Need to include scheduling information.

> If you use the latest version of Anki, please check the box `Support older Anki versions (slower/larger files)` when you export.

You can export it via `File -> Export...` or `Ctrl + E` in the main window of Anki.

Then replace the `filename` with yours in the next code cell. And set the `timezone` and `next_day_starts_at` which can be found in your preferences of Anki.

After that, just run all (`Runtime -> Run all` or `Ctrl + F9`) and wait for minutes. You can see the optimal parameters in section **2.3 Result**. Copy them, replace the parameters in `fsrs4anki_scheduler.js`, and paste them into the custom scheduling of your deck options (require Anki version >= 2.1.55).

**NOTE**: The default output is generated from my review logs. If you find the output is the same as mine, maybe your notebook hasn't run there.

**Contribute to SRS Research**: If you want to share your data with me, please fill this form: https://forms.gle/KaojsBbhMCytaA7h8

In [19]:
# Here are some settings that you need to replace before running this optimizer.

filename = "Anki-FSRS-05-12-26-Export-TrueSuspendCard.colpkg"
# If you upload deck file, replace it with your deck filename. E.g., ALL__Learning.apkg
# If you upload collection file, replace it with your colpkg filename. E.g., collection-2022-09-18@13-21-58.colpkg

# Replace it with your timezone. I'm in China, so I use Asia/Shanghai.
# You can find your timezone here: https://gist.github.com/heyalexej/8bf688fd67d7199be4a1682b3eec7568
timezone = 'US/Central'

# Replace it with your Anki's setting in Preferences -> Scheduling.
next_day_starts_at = 2

# Replace it if you don't want the optimizer to use the review logs before a specific date.
revlog_start_date = "2006-10-05"  # YYYY-MM-DD

# Set it to True if you don't want the optimizer to use the review logs from suspended cards.
filter_out_suspended_cards = True

# Red: 1, Orange: 2, Green: 3, Blue: 4, Pink: 5, Turquoise: 6, Purple: 7
# Set it to [1, 2] if you don't want the optimizer to use the review logs from cards with red or orange flag.
filter_out_flags = []

enable_short_term = True

recency_weight = True

## 1 Build dataset

### 1.1 Extract Anki collection & deck file

In [20]:
%pip install -q fsrs_optimizer==6.1.5
# for local development
# import os
# import sys
# sys.path.insert(0, os.path.abspath('../fsrs-optimizer/src/fsrs_optimizer/'))
import fsrs_optimizer as optimizer
optimizer = optimizer.Optimizer(enable_short_term=enable_short_term)
optimizer.anki_extract(filename, filter_out_suspended_cards, filter_out_flags)

Deck file extracted successfully!
revlog.csv saved.


### 1.2 Create time-series feature & analysis

The following code cell will extract the review logs from your Anki collection and preprocess them to a trainset which is saved in [./revlog_history.tsv](./revlog_history.tsv).

The time-series features are important in optimizing the model's parameters. For more detail, please see my paper: https://www.maimemo.com/paper/

Then it will generate a concise analysis for your review logs.

- The `r_history` is the history of ratings on each review. `3,3,3,1` means that you press `Good, Good, Good, Again`. It only contains the first rating for each card on the review date, i.e., when you press `Again` in review and  `Good` in relearning steps 10min later, only `Again` will be recorded.
- The `avg_interval` is the actual average interval after you rate your cards as the `r_history`. It could be longer than the interval given by Anki's built-in scheduler because you reviewed some overdue cards.
- The `avg_retention` is the average retention after you press as the `r_history`. `Again` counts as failed recall, and `Hard, Good and Easy` count as successful recall. Retention is the percentage of your successful recall.
- The `stability` is the estimated memory state variable, which is an approximate interval that leads to 90% retention.
- The `factor` is `stability / previous stability`.
- The `group_cnt` is the number of review logs that have the same `r_history`.

In [21]:
analysis = optimizer.create_time_series(
    timezone, revlog_start_date, next_day_starts_at)
print(analysis)

  0%|          | 0/1871 [00:00<?, ?it/s]

Trainset saved.
Retention calculated.


  0%|          | 0/4517 [00:00<?, ?it/s]

Stability calculated.


analysis:   0%|          | 0/123 [00:00<?, ?it/s]

Analysis saved!
1:again, 2:hard, 3:good, 4:easy
first_rating  i       r_history  avg_interval  avg_retention  stability  factor  group_cnt
           1  2             (1)           1.1          0.316        0.0     NaN        770
           1  3           (1),3           1.2          0.875        1.0     inf        210
           1  4         (1),3,3           3.5          0.954       11.3   11.30        178
           1  5       (1),3,3,3          10.5          0.926       16.7    1.48        167
           1  6     (1),3,3,3,3          27.6          0.970       77.8    4.66        150
           1  7   (1),3,3,3,3,3          56.8          0.943      128.7    1.65        118
           3  2             (3)           1.2          0.891        1.3     inf        688
           3  2           (3,3)           7.9          0.915        8.0     inf        113
           3  3           (3),3           5.6          0.967       17.7   13.62        587
           3  3         (3,3),3          1

## 2 Optimize parameter

### 2.1 Define & Train the model

FSRS is a time-series model for predicting memory states.

The [./revlog_history.tsv](./revlog_history.tsv) generated before will be used for training the FSRS model.

In [22]:
optimizer.define_model()
optimizer.pretrain(verbose=False)
optimizer.train(verbose=False, recency_weight=recency_weight)

  0%|          | 0/14067 [00:00<?, ?it/s]

[]

### 2.2 Result

Copy the optimal parameters for FSRS for you in the output of next code cell after running.

In [23]:
print(optimizer.w)

[0.0335, 0.1304, 1.8676, 11.0526, 6.2525, 0.7839, 3.3376, 0.0775, 1.9972, 0.2906, 0.8962, 1.3623, 0.0844, 0.1739, 1.4817, 0.6056, 1.8527, 1.0114, 0.4915, 0.3717, 0.3702]


<font color=orange>Note: These values should be used with build-in FSRS of Anki 23.12 or custom scheduling script of FSRS4Anki v4.11.0</font>

### 2.3 Preview

You can see the memory states and intervals generated by FSRS as if you press the good in each review at the due date scheduled by FSRS.

In [24]:
requestRetention = 0.95  # recommended setting: 0.8 ~ 0.9

preview = optimizer.preview(requestRetention)
print(preview)

1:again, 2:hard, 3:good, 4:easy

first rating: 1
rating history: (1,3,3),3,3,3,3,3,3,3,3
interval history: 0.0d,0.0d,0.0d,1.0d,2.0d,5.0d,10.0d,20.0d,1.2m,2.1m,3.4m
factor history: 0.0,0.0,0.0,0.0,2.00,2.50,2.00,2.00,1.80,1.72,1.65
difficulty history: 0,6.3,5.5,4.8,4.2,3.6,3.1,2.6,2.2,1.7,1.3
stability history: 0,0.0,0.2,0.6,5.1,11.5,25.1,48.5,89.4,154.4,254.6

first rating: 2
rating history: (2,3,3),3,3,3,3,3,3,3,3
interval history: 0.0d,0.0d,0.0d,1.0d,2.0d,5.0d,11.0d,22.0d,1.4m,2.4m,4.0m
factor history: 0.0,0.0,0.0,0.0,2.00,2.50,2.20,2.00,1.86,1.76,1.67
difficulty history: 0,5.1,4.4,3.8,3.3,2.8,2.3,1.9,1.5,1.1,1.0
stability history: 0,0.1,0.5,1.0,6.0,13.0,27.7,55.2,102.4,179.1,298.0

first rating: 3
rating history: (3,3),3,3,3,3,3,3,3,3,3
interval history: 0.0d,0.0d,1.0d,3.0d,8.0d,18.0d,1.2m,2.2m,3.7m,5.9m,9.0m
factor history: 0.0,0.0,0.0,3.00,2.67,2.25,2.00,1.83,1.68,1.59,1.52
difficulty history: 0,2.5,2.0,1.6,1.2,1.0,1.0,1.0,1.0,1.0,1.0
stability history: 0,1.9,2.4,7.7,19.4,44.3,89.

You can change the `test_rating_sequence` to see the scheduling intervals in different ratings.

In [15]:
test_rating_sequence = "3,3,3,3,3,1,1,3,3,3,3,3"
requestRetention = 0.9  # recommended setting: 0.8 ~ 0.9

preview_sequence = optimizer.preview_sequence(
    test_rating_sequence, requestRetention)
print(preview_sequence)

rating history: 3,3,3,3,3,1,1,3,3,3,3,3
interval history: 0.0d,2.0d,10.0d,1.2m,3.7m,8.9m,3.0d,1.0d,3.0d,10.0d,1.1m,2.9m,7.2m
factor history: 0.0,0.0,5.00,3.70,2.97,2.43,0.01,0.33,3.00,3.33,3.20,2.75,2.44
difficulty history: 0,2.5,1.5,1.0,1.0,1.0,5.6,6.7,5.0,3.6,2.5,1.5,1.0


### 2.4 Predict memory states and distribution of difficulty

Predict memory states for each review group and save them in [./prediction.tsv](./prediction.tsv).

Meanwhile, it will count the distribution of difficulty.

In [17]:
optimizer.predict_memory_states()

,count
difficulty,
1,0.344158
2,0.127034
3,0.124633
4,0.066885
5,0.063750
6,0.092358
7,0.181182
